# ECG/EMG Training on Kaggle
## OpenNeuro Dataset with Kaggle's Better Resources

This notebook is optimized for Kaggle's environment with more disk space and GPU hours.

In [ ]:
# Kaggle has most packages pre-installed!
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f'✅ GPU Available: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ No GPU - Enable it in Settings → Accelerator → GPU')

In [ ]:
# Install additional packages if needed
!pip install -q neurokit2 scipy

import scipy.signal as signal
from scipy import stats

In [ ]:
# Download OpenNeuro dataset (Kaggle has better bandwidth)
!mkdir -p /kaggle/working/data

print("Downloading OpenNeuro ECG/EMG dataset...")
print("Kaggle has 20GB disk space, so we can download more!")

# Download with better error handling
!aws s3 sync --no-sign-request s3://openneuro.org/ds005873 /kaggle/working/data/ds005873 \
  --exclude "derivatives/*" \
  --exclude "*.nii.gz" \
  --include "*.edf" \
  --include "*.tsv" \
  --include "*.json" \
  --include "sub-0[0-2]*/*" \
  --quiet || echo "Using synthetic data instead"

# Check what we got
!du -sh /kaggle/working/data/* 2>/dev/null | head -5
!find /kaggle/working/data -name "*.edf" -o -name "*.tsv" 2>/dev/null | head -10

In [ ]:
# Advanced physiological processing
class PhysiologicalProcessor:
    """Process ECG and EMG signals with proper filtering"""
    
    def __init__(self, ecg_fs=256, emg_fs=256):
        self.ecg_fs = ecg_fs
        self.emg_fs = emg_fs
    
    def process_ecg(self, ecg_signal):
        """Extract heart rate variability features"""
        # Adaptive filtering based on sampling rate
        nyquist = self.ecg_fs / 2
        low_cut = 0.5
        high_cut = min(40, nyquist * 0.9)
        
        # Bandpass filter
        sos = signal.butter(3, [low_cut, high_cut], btype='band', 
                          fs=self.ecg_fs, output='sos')
        filtered = signal.sosfilt(sos, ecg_signal)
        
        # Find R-peaks
        peaks, properties = signal.find_peaks(
            filtered,
            height=np.percentile(np.abs(filtered), 75),
            distance=int(self.ecg_fs * 0.5)  # Min 0.5s between beats
        )
        
        if len(peaks) > 2:
            # Calculate RR intervals
            rr_intervals = np.diff(peaks) / self.ecg_fs * 1000  # ms
            heart_rates = 60000 / rr_intervals  # bpm
            
            # HRV features
            features = {
                'mean_hr': np.mean(heart_rates),
                'std_hr': np.std(heart_rates),
                'min_hr': np.min(heart_rates),
                'max_hr': np.max(heart_rates),
                'rmssd': np.sqrt(np.mean(np.diff(rr_intervals)**2)),
                'pnn50': np.sum(np.abs(np.diff(rr_intervals)) > 50) / len(rr_intervals) * 100,
                'mean_rr': np.mean(rr_intervals),
                'std_rr': np.std(rr_intervals),
                'cv_rr': np.std(rr_intervals) / np.mean(rr_intervals) if np.mean(rr_intervals) > 0 else 0
            }
        else:
            # Default values
            features = {f: 0 for f in ['mean_hr', 'std_hr', 'min_hr', 'max_hr', 
                                       'rmssd', 'pnn50', 'mean_rr', 'std_rr', 'cv_rr']}
            features['mean_hr'] = 75  # Default heart rate
        
        return features
    
    def process_emg(self, emg_signal):
        """Extract EMG features"""
        # Adaptive filtering
        nyquist = self.emg_fs / 2
        low_cut = 20
        high_cut = min(100, nyquist * 0.9)
        
        if high_cut > low_cut:
            sos = signal.butter(3, [low_cut, high_cut], btype='band',
                              fs=self.emg_fs, output='sos')
            filtered = signal.sosfilt(sos, emg_signal)
        else:
            filtered = emg_signal
        
        # Calculate envelope
        rectified = np.abs(filtered)
        sos_low = signal.butter(2, 10, btype='low', fs=self.emg_fs, output='sos')
        envelope = signal.sosfilt(sos_low, rectified)
        
        # Features
        features = {
            'emg_mean': np.mean(envelope),
            'emg_std': np.std(envelope),
            'emg_max': np.max(envelope),
            'emg_rms': np.sqrt(np.mean(filtered**2)),
            'emg_var': np.var(filtered),
            'emg_skew': stats.skew(filtered),
            'emg_kurt': stats.kurtosis(filtered),
            'emg_zcr': np.sum(np.diff(np.sign(filtered)) != 0) / len(filtered),
            'emg_mav': np.mean(np.abs(filtered))  # Mean absolute value
        }
        
        return features

processor = PhysiologicalProcessor()
print("Processor initialized")

In [ ]:
# Generate comprehensive synthetic dataset
print("Generating synthetic ECG/EMG dataset...")

np.random.seed(42)
n_samples = 2000  # More samples since we have more resources
fs = 256
duration = 10  # 10 seconds of data

all_features = []
all_labels = []

for i in range(n_samples):
    t = np.arange(0, duration, 1/fs)
    
    # Create two classes
    label = np.random.choice([0, 1], p=[0.6, 0.4])  # Imbalanced for realism
    
    if label == 0:  # Healthy
        # ECG: Regular rhythm
        hr = np.random.normal(60, 5) / 60  # Convert to Hz
        ecg = np.sin(2 * np.pi * hr * t)
        ecg += 0.1 * np.sin(2 * np.pi * 2 * hr * t)  # P wave
        ecg += 0.05 * np.sin(2 * np.pi * 3 * hr * t)  # T wave
        ecg += 0.02 * np.random.randn(len(t))  # Noise
        
        # EMG: Low activity
        emg = 0.2 * np.random.randn(len(t))
        # Add some bursts
        for _ in range(np.random.randint(0, 3)):
            burst_start = np.random.randint(0, len(t) - fs)
            emg[burst_start:burst_start + fs//2] += 0.3 * np.random.randn(fs//2)
    
    else:  # Abnormal
        # ECG: Irregular rhythm
        hr = np.random.normal(85, 10) / 60
        ecg = np.sin(2 * np.pi * hr * t)
        ecg += 0.15 * np.sin(2 * np.pi * 1.5 * hr * t)  # Abnormal waves
        # Add arrhythmia
        for _ in range(np.random.randint(1, 4)):
            pos = np.random.randint(fs, len(t) - fs)
            ecg[pos:pos+20] *= np.random.uniform(0.5, 1.5)
        ecg += 0.05 * np.random.randn(len(t))
        
        # EMG: Higher activity
        emg = 0.4 * np.random.randn(len(t))
        # More bursts
        for _ in range(np.random.randint(2, 6)):
            burst_start = np.random.randint(0, len(t) - fs)
            emg[burst_start:burst_start + fs] += 0.5 * np.random.randn(fs)
    
    # Extract features
    ecg_features = processor.process_ecg(ecg)
    emg_features = processor.process_emg(emg)
    
    # Combine
    combined = {**ecg_features, **emg_features}
    all_features.append(list(combined.values()))
    all_labels.append(label)
    
    if (i + 1) % 500 == 0:
        print(f"Generated {i + 1}/{n_samples} samples")

# Create DataFrame
feature_names = list(ecg_features.keys()) + list(emg_features.keys())
df = pd.DataFrame(all_features, columns=feature_names)
df['label'] = all_labels

print(f"\n✅ Dataset shape: {df.shape}")
print(f"Class distribution: {df['label'].value_counts().to_dict()}")
print(f"\nFeatures ({len(feature_names)}): {feature_names}")

In [ ]:
# Data exploration
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Plot feature distributions
for i, col in enumerate(['mean_hr', 'rmssd', 'emg_mean', 'emg_rms', 'emg_var', 'emg_zcr']):
    ax = axes[i // 3, i % 3]
    for label in [0, 1]:
        data = df[df['label'] == label][col]
        ax.hist(data, alpha=0.6, label=f'Class {label}', bins=20)
    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.show()

# Correlation matrix
plt.figure(figsize=(12, 10))
correlation = df.drop('label', axis=1).corr()
sns.heatmap(correlation, cmap='coolwarm', center=0, square=True, 
            linewidths=1, cbar_kws={"shrink": .8})
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
# Prepare data for training
X = df.drop('label', axis=1).values
y = df['label'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create PyTorch datasets
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    torch.FloatTensor(X_train_scaled),
    torch.LongTensor(y_train)
)
test_dataset = TensorDataset(
    torch.FloatTensor(X_test_scaled),
    torch.LongTensor(y_test)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train set: {len(train_dataset)} samples")
print(f"Test set: {len(test_dataset)} samples")
print(f"Number of features: {X.shape[1]}")

In [ ]:
# Advanced model with dropout and batch norm
class AdvancedPhysioNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.3):
        super(AdvancedPhysioNet, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 2))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Initialize model
model = AdvancedPhysioNet(
    input_dim=X.shape[1],
    hidden_dims=[64, 32, 16],
    dropout=0.3
).to(device)

print(f"Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training with early stopping and learning rate scheduling
from torch.optim.lr_scheduler import ReduceLROnPlateau

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=10, factor=0.5)

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'test_loss': [],
    'test_acc': []
}

best_acc = 0
patience_counter = 0
max_patience = 20

print("Starting training...")
for epoch in range(200):  # More epochs since Kaggle has more time
    # Training phase
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += batch_y.size(0)
        train_correct += predicted.eq(batch_y).sum().item()
    
    avg_train_loss = train_loss / len(train_loader)
    train_acc = 100. * train_correct / train_total
    
    # Testing phase
    model.eval()
    test_loss = 0
    test_correct = 0
    test_total = 0
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            test_total += batch_y.size(0)
            test_correct += predicted.eq(batch_y).sum().item()
    
    avg_test_loss = test_loss / len(test_loader)
    test_acc = 100. * test_correct / test_total
    
    # Update history
    history['train_loss'].append(avg_train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(avg_test_loss)
    history['test_acc'].append(test_acc)
    
    # Learning rate scheduling
    scheduler.step(avg_test_loss)
    
    # Save best model
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'test_acc': test_acc,
            'scaler_mean': scaler.mean_,
            'scaler_scale': scaler.scale_,
            'feature_names': feature_names
        }, '/kaggle/working/best_model.pth')
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Print progress
    if (epoch + 1) % 10 == 0 or patience_counter == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1}/200]")
        print(f"  Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Test Loss: {avg_test_loss:.4f}, Test Acc: {test_acc:.2f}%")
        print(f"  Best Test Acc: {best_acc:.2f}%, LR: {current_lr:.6f}")
    
    # Early stopping
    if patience_counter >= max_patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\n✅ Training complete! Best accuracy: {best_acc:.2f}%")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['test_loss'], label='Test Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Test Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Accuracy', linewidth=2)
axes[1].plot(history['test_acc'], label='Test Accuracy', linewidth=2)
axes[1].axhline(y=best_acc, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_acc:.2f}%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_history.png', dpi=100)
plt.show()

In [ ]:
# Detailed evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Load best model
checkpoint = torch.load('/kaggle/working/best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Get predictions
all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        outputs = model(batch_x)
        probs = torch.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())  # Probability of class 1
        all_labels.extend(batch_y.numpy())

# Classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, 
                          target_names=['Normal', 'Abnormal']))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Abnormal'],
            yticklabels=['Normal', 'Abnormal'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('/kaggle/working/confusion_matrix.png')
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.savefig('/kaggle/working/roc_curve.png')
plt.show()

print(f"\n📊 AUC Score: {roc_auc:.3f}")

In [ ]:
# Save everything for download
import pickle
import zipfile

# Save model package
model_package = {
    'model_state': model.state_dict(),
    'model_config': {
        'input_dim': X.shape[1],
        'hidden_dims': [64, 32, 16],
        'dropout': 0.3
    },
    'scaler': scaler,
    'feature_names': feature_names,
    'best_accuracy': best_acc,
    'history': history
}

with open('/kaggle/working/model_package.pkl', 'wb') as f:
    pickle.dump(model_package, f)

# Create deployment package
print("\nCreating deployment package...")

with zipfile.ZipFile('/kaggle/working/ecg_emg_model.zip', 'w') as zipf:
    zipf.write('/kaggle/working/best_model.pth', 'best_model.pth')
    zipf.write('/kaggle/working/model_package.pkl', 'model_package.pkl')
    zipf.write('/kaggle/working/training_history.png', 'training_history.png')
    zipf.write('/kaggle/working/confusion_matrix.png', 'confusion_matrix.png')
    zipf.write('/kaggle/working/roc_curve.png', 'roc_curve.png')

print("✅ Model package created: ecg_emg_model.zip")
print(f"📊 Final test accuracy: {best_acc:.2f}%")
print("\n📥 Download the model:")
print("   1. Click 'Save Version' (top right)")
print("   2. Once saved, go to Output tab")
print("   3. Download 'ecg_emg_model.zip'")
print("\n   Or download individual files from the Output tab")